In [1]:
import QuantLib as ql

<h2>A simple example</h2>
Let’s say you have a bond which has already matured; for instance, this one.

In [2]:
bond = ql.ZeroCouponBond(
    settlementDays=3,
    calendar=ql.TARGET(),
    faceAmount=10_000,
    maturityDate=ql.Date(8, ql.February, 2025)
)

As I write this, it’s April 2025; but let’s say you’re checking some end-of-year figures, and want to reprice the bond as of that date:

In [3]:
as_of_date = ql.Date(31, ql.December, 2024)

Next, you set up a discount curve with the correct reference date and use it to price the bond by passing it to the usual engine.

In [4]:
discount_curve = ql.RelinkableYieldTermStructureHandle(
    ql.FlatForward(as_of_date, 0.02, ql.Actual360())
)
engine = ql.DiscountingBondEngine(discount_curve)
bond.setPricingEngine(engine)

You already know where this is going: you ask for the price, and you get a null one.

In [5]:
bond.cleanPrice()

0.0

In [6]:
bond.isExpired()

True

The correct way to get its end-of-year price, of course, is to set the evaluation date:

In [7]:
ql.Settings.instance().evaluationDate=as_of_date

In [8]:
bond.cleanPrice()

99.80574447629698

<h2>Term Structure</h2>
See file <a href="TermStructuresAndTheirReferenceDates.ipynb">TermStructuresAndTheirReferenceDates</a>

<h2>Fixings</h2>

Another case where the evaluation date plays a role is determining whether the fixing of an interest-rate index should be calculated. Let’s create an index and give it a flat 1% forecast curve:

In [10]:
forecast_curve = ql.YieldTermStructureHandle(
    ql.FlatForward(0, ql.NullCalendar(), 0.01, ql.Actual360())
    
)

In [11]:
index=ql.Euribor3M(forecast_curve)

The evaluation date is still at the end of 2024. If we try to ask the index for a fixing at a past date, it’s going to gently but firmly rebuke us:

In [12]:
try:
    index.fixing(ql.Date(15, ql.February, 2021))
except Exception as e:
    print(f"{type(e).__name__}:{e}")

RuntimeError:Missing Euribor3M Actual/360 fixing for February 15th, 2021


That’s because the fixing is in the past with respect to the evaluation date, and therefore it can’t be forecast off the curve: it must have been stored instead (here I’ll use a bogus 2% value). Once we do this, the index returns the stored value.

In [13]:
index.addFixing(ql.Date(15, ql.February, 2021), 0.02)

In [14]:
index.fixing(ql.Date(15, ql.February, 2021))

0.02

If I move the evaluation date before that fixing date, though, the behavior of the index changes. Now the fixing date is in the future with respect to the evaluation date, and therefore the fixing is forecast from the 1% curve we set previously, even though the stored value is still there:

In [15]:
ql.Settings.instance().evaluationDate=ql.Date(1, ql.February, 2021)

In [16]:
index.fixing(ql.Date(15, ql.February, 2021))

0.010012371303880592